# Dot Product & Cosine Similarity

The dot product is the core operation behind both **attention** (Q·K) and **RAG retrieval** (query vs document similarity).

If you understand dot product, you understand how transformers "look at" tokens and how vector search finds relevant documents.

## 1. Dot Product

**Formula:**

```
A · B = a₁b₁ + a₂b₂ + ... + aₙbₙ
```

Multiply matching elements, then sum.

**What it measures:** How much two vectors point in the same direction.
- Large positive → same direction (similar)
- Zero → perpendicular (unrelated)
- Large negative → opposite directions (dissimilar)

In [ ]:
def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

a = [1, 2, 3]
b = [4, 5, 6]

print("Dot Product")
print("-" * 35)
print(f"A = {a}")
print(f"B = {b}")
print(f"A · B = {a[0]}×{b[0]} + {a[1]}×{b[1]} + {a[2]}×{b[2]}")
print(f"      = {a[0]*b[0]} + {a[1]*b[1]} + {a[2]*b[2]}")
print(f"      = {dot_product(a, b)}")

## 2. Step by Step — What the Number Means

In [ ]:
import math

# Three cases: same direction, perpendicular, opposite

cases = [
    ("Same direction",      [1, 0],  [1, 0]),
    ("Similar direction",   [1, 0],  [0.7, 0.7]),
    ("Perpendicular",       [1, 0],  [0, 1]),
    ("Somewhat opposite",   [1, 0],  [-0.7, 0.7]),
    ("Opposite direction",  [1, 0],  [-1, 0]),
]

print("What dot product values mean (2D vectors)")
print("=" * 55)
print(f"{'Relationship':<22} {'A':>8} {'B':>12} {'A·B':>6}")
print("-" * 55)

for name, a, b in cases:
    dp = dot_product(a, b)
    print(f"{name:<22} {str(a):>8} {str(b):>12} {dp:>6.2f}")

print(f"\nPositive → similar, Zero → unrelated, Negative → opposite")

## 3. Cosine Similarity

**Problem with raw dot product:** It depends on vector **length**, not just direction.

```
[1, 0] · [1, 0] = 1
[100, 0] · [100, 0] = 10000   ← same direction, but 10000x larger!
```

**Cosine similarity** fixes this by normalizing:

```
cos(θ) = (A · B) / (|A| × |B|)
```

- Output range: **-1 to 1** (always, regardless of vector size)
- 1 = identical direction
- 0 = perpendicular
- -1 = opposite direction

In [ ]:
def magnitude(v):
    return math.sqrt(sum(x ** 2 for x in v))

def cosine_similarity(a, b):
    dot = dot_product(a, b)
    mag_a = magnitude(a)
    mag_b = magnitude(b)
    if mag_a == 0 or mag_b == 0:
        return 0
    return dot / (mag_a * mag_b)

# Show why normalization matters
print("Why cosine similarity > raw dot product")
print("=" * 60)

a_small = [1, 2]
a_big = [100, 200]  # same direction, 100x longer
b = [2, 1]

print(f"{'Vectors':<30} {'Dot':>8} {'Cosine':>8}")
print("-" * 50)
print(f"{'[1,2] vs [2,1]':<30} {dot_product(a_small, b):>8.1f} {cosine_similarity(a_small, b):>8.4f}")
print(f"{'[100,200] vs [2,1]':<30} {dot_product(a_big, b):>8.1f} {cosine_similarity(a_big, b):>8.4f}")

print(f"\nDot product: 400 vs 4 — suggests [100,200] is 100x more similar")
print(f"Cosine: 0.8 vs 0.8 — correctly shows they're equally similar")
print(f"\nThe direction is the same; only the length differs.")
print(f"Cosine similarity ignores length and measures only direction.")

In [ ]:
# Step by step cosine similarity

a = [3, 4]
b = [4, 3]

dot = dot_product(a, b)
mag_a = magnitude(a)
mag_b = magnitude(b)
cos_sim = cosine_similarity(a, b)

print("Step by step cosine similarity")
print("-" * 45)
print(f"A = {a}")
print(f"B = {b}")
print(f"")
print(f"Step 1: Dot product")
print(f"  A·B = {a[0]}×{b[0]} + {a[1]}×{b[1]} = {dot}")
print(f"")
print(f"Step 2: Magnitudes")
print(f"  |A| = sqrt({a[0]}² + {a[1]}²) = sqrt({a[0]**2 + a[1]**2}) = {mag_a:.2f}")
print(f"  |B| = sqrt({b[0]}² + {b[1]}²) = sqrt({b[0]**2 + b[1]**2}) = {mag_b:.2f}")
print(f"")
print(f"Step 3: Divide")
print(f"  cos(θ) = {dot} / ({mag_a:.2f} × {mag_b:.2f})")
print(f"         = {dot} / {mag_a * mag_b:.2f}")
print(f"         = {cos_sim:.4f}")
print(f"")
print(f"  0.96 is close to 1.0 → very similar direction")

## 4. ASCII Visualization

In [ ]:
# Show vectors as arrows on a 2D grid

def draw_vectors(pairs, title):
    size = 11
    center = size // 2
    grid = [[' ' for _ in range(size)] for _ in range(size)]
    
    # Draw axes
    for i in range(size):
        grid[center][i] = '-'
        grid[i][center] = '|'
    grid[center][center] = '+'
    
    # Draw vectors
    for label, vec, char in pairs:
        steps = 4
        for t in range(1, steps + 1):
            px = center + round(vec[0] * t / max(abs(vec[0]), abs(vec[1]), 1) * (steps))
            py = center - round(vec[1] * t / max(abs(vec[0]), abs(vec[1]), 1) * (steps))
            if 0 <= px < size and 0 <= py < size:
                grid[py][px] = char
    
    cos = cosine_similarity(pairs[0][1], pairs[1][1])
    print(f"\n{title}  (cosine = {cos:.2f})")
    for row in grid:
        print('  ' + ''.join(row))
    for label, vec, char in pairs:
        print(f"  {char} = {label} {vec}")

draw_vectors(
    [("A", [1, 1], 'A'), ("B", [1, 0.8], 'B')],
    "Very similar (almost parallel)"
)

draw_vectors(
    [("A", [1, 0], 'A'), ("B", [0, 1], 'B')],
    "Perpendicular (unrelated)"
)

draw_vectors(
    [("A", [1, 1], 'A'), ("B", [-1, -1], 'B')],
    "Opposite (dissimilar)"
)

## 5. Why This Matters for RAG

RAG retrieval = finding the document whose embedding vector is most similar to the query embedding.

This is literally cosine similarity (or dot product) between vectors.

In [ ]:
# Simulating RAG retrieval with cosine similarity
# In reality, embeddings are 768-3072 dimensions.
# We use 5 dimensions here to keep it visible.

# Pretend embedding dimensions represent:
# [programming, cooking, sports, science, music]

query = {
    "text": "How to train a neural network?",
    "embedding": [0.9, 0.0, 0.0, 0.8, 0.0]
}

documents = [
    {"text": "Backpropagation tutorial",     "embedding": [0.85, 0.0, 0.0, 0.9, 0.0]},
    {"text": "Python cookbook recipes",        "embedding": [0.5, 0.8, 0.0, 0.1, 0.0]},
    {"text": "Football training drills",      "embedding": [0.0, 0.0, 0.9, 0.1, 0.0]},
    {"text": "Gradient descent explained",    "embedding": [0.8, 0.0, 0.0, 0.85, 0.0]},
    {"text": "Guitar chord progressions",     "embedding": [0.0, 0.0, 0.0, 0.1, 0.95]},
]

print(f'Query: "{query["text"]}"')
print(f"Query embedding: {query['embedding']}")
print(f"Dimensions: [programming, cooking, sports, science, music]")
print("=" * 65)

results = []
for doc in documents:
    sim = cosine_similarity(query["embedding"], doc["embedding"])
    results.append((doc["text"], sim, doc["embedding"]))

results.sort(key=lambda x: -x[1])

print(f"\n{'Rank':<5} {'Document':<30} {'Cosine':>7}  Relevance")
print("-" * 65)
for i, (text, sim, emb) in enumerate(results, 1):
    bar = '█' * int(sim * 20)
    marker = " ← retrieve this" if i <= 2 else ""
    print(f"{i:<5} {text:<30} {sim:>7.4f}  {bar}{marker}")

print(f"\nRAG picks the top-k most similar documents.")
print(f"The dot product / cosine similarity IS the retrieval.")

## 6. Why This Matters for Attention

In attention: **Q · K** is a dot product that measures how much each token should attend to every other token.

```
Attention(Q, K, V) = softmax(Q · K^T / √d) × V
                      ↑
                  dot product is here
```

In [ ]:
# Simplified attention: Q·K gives attention scores

tokens = ["I", "love", "cats"]

# Pretend Q and K vectors (3 dimensions for visibility)
Q = {
    "I":    [0.2, 0.8, 0.1],
    "love": [0.9, 0.3, 0.7],
    "cats": [0.1, 0.6, 0.9],
}
K = {
    "I":    [0.3, 0.7, 0.2],
    "love": [0.8, 0.2, 0.6],
    "cats": [0.2, 0.5, 0.8],
}

print('Attention scores: Q · K for each token pair')
print("=" * 50)

# Compute Q·K for each query-key pair
print(f"\n{'Query':<8}", end="")
for k_token in tokens:
    print(f"{'K=' + k_token:>10}", end="")
print()
print("-" * 40)

for q_token in tokens:
    print(f"{q_token:<8}", end="")
    for k_token in tokens:
        score = dot_product(Q[q_token], K[k_token])
        print(f"{score:>10.3f}", end="")
    print()

print(f"\nHigher score = this query pays more attention to this key.")
print(f"These raw scores go through softmax to become attention weights.")
print(f"(See softmax.ipynb and attention_qkv.ipynb for the full picture)")

## 7. Euclidean Distance vs Cosine Similarity

Two ways to measure "how similar are these vectors?"

| | Cosine Similarity | Euclidean Distance |
|---|---|---|
| **Measures** | Direction (angle) | Position (straight-line distance) |
| **Range** | -1 to 1 | 0 to ∞ |
| **Similar =** | Close to 1 | Close to 0 |
| **Affected by length?** | No | Yes |
| **Best for** | Semantic similarity (meaning) | Spatial distance (location) |

In [ ]:
def euclidean_distance(a, b):
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))

# Show when they disagree

query =     [1, 1]
doc_short = [2, 2]       # same direction, close
doc_long =  [100, 100]   # same direction, far away
doc_perp =  [2, 0]       # different direction, close

print("Cosine vs Euclidean: when they disagree")
print(f"Query: {query}")
print("=" * 60)
print(f"{'Document':<16} {'Cosine':>8} {'Euclidean':>10}  Better match by...")
print("-" * 60)

docs = [
    ("[2, 2]", doc_short),
    ("[100, 100]", doc_long),
    ("[2, 0]", doc_perp),
]

for name, doc in docs:
    cos = cosine_similarity(query, doc)
    euc = euclidean_distance(query, doc)
    print(f"{name:<16} {cos:>8.4f} {euc:>10.2f}")

print(f"\n[100,100] has cosine=1.0 (perfect match!) but euclidean=138")
print(f"[2, 0] has euclidean=1.41 (very close!) but cosine=0.71")
print(f"\nFor RAG / embeddings: use COSINE (direction = meaning)")
print(f"For k-nearest neighbors in physical space: use EUCLIDEAN")

## 8. NumPy Version (Vectorized)

In [ ]:
import numpy as np

def dot_product_np(a, b):
    return np.dot(a, b)

def cosine_similarity_np(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("NumPy vectorized")
print(f"A = {a}")
print(f"B = {b}")
print(f"Dot product:       {dot_product_np(a, b)}")
print(f"Cosine similarity: {cosine_similarity_np(a, b):.4f}")

# Batch: compute similarity of query against many documents at once
query = np.array([0.9, 0.0, 0.8])
docs = np.array([
    [0.85, 0.0, 0.9],
    [0.0, 0.9, 0.1],
    [0.8, 0.0, 0.85],
])

# All similarities in one operation
scores = docs @ query  # matrix-vector dot product
norms = np.linalg.norm(docs, axis=1) * np.linalg.norm(query)
cosines = scores / norms

print(f"\nBatch cosine similarity (query vs 3 docs):")
print(f"  {np.round(cosines, 4)}")
print(f"  Best match: doc {np.argmax(cosines)}")

## Summary

| | Dot Product | Cosine Similarity |
|---|---|---|
| **Formula** | `Σ aᵢbᵢ` | `(A·B) / (\|A\| × \|B\|)` |
| **Range** | -∞ to ∞ | -1 to 1 |
| **Measures** | Direction + magnitude | Direction only |
| **Affected by length?** | Yes | No |

### Where they appear

| Context | Operation | Why |
|---|---|---|
| **Attention (Q·K)** | Dot product | Measures how much one token should focus on another |
| **RAG retrieval** | Cosine similarity | Finds documents with similar meaning regardless of length |
| **Embedding search** | Cosine similarity | Standard metric for vector databases |
| **Training loss** | Dot product | Used inside cross-entropy and other loss functions |